In [105]:
import requests
from requests.auth import HTTPBasicAuth
import json

JIRA_URL = "https://<your-url>.atlassian.net"
API_SEARCH_ENDPOINT = "/rest/api/3/search"  # Changed endpoint
USERNAME = "your-email@something.com"
API_TOKEN = "your"

headers = {
    "Accept": "application/json"
}


### Getting all issues from the multi-repos project

In [43]:
def get_labeled_issues(project, labels, start_at=0, max_results=50):
    """
    Get issues with specified labels in a given project using JQL search
    """
    labels_query = " AND ".join([f'labels = "{label}"' for label in labels])
    jql_query = f'project = {project} AND {labels_query}'
    
    url = f"{JIRA_URL}{API_SEARCH_ENDPOINT}"
    
    params = {
        "jql": jql_query,
        "startAt": start_at,
        "maxResults": max_results,
        "fields": [
            "key",
            "summary",
            "status",
            "labels",
            "assignee",
            "created",
            "updated"
        ]
    }
    
    try:
        response = requests.get(
            url,
            headers=headers,
            params=params,
            auth=HTTPBasicAuth(USERNAME, API_TOKEN)
        )
        response.raise_for_status()
        return response.json()
    except requests.exceptions.RequestException as e:
        print(f"Error fetching issues: {e}")
        return None

def get_all_labeled_issues(project, labels):
    """
    Get all issues using pagination
    """
    all_issues = []
    start_at = 0
    max_results = 50
    
    while True:
        result = get_labeled_issues(project, labels, start_at, max_results)
        
        if not result or 'issues' not in result:
            break
            
        issues = result['issues']
        all_issues.extend(issues)
        
        if len(issues) < max_results:
            break
            
        start_at += max_results
    
    return all_issues


In [44]:
# Example usage
project = "<The project ID>"
labels = ["<labels you want to filter>"]
issues = get_all_labeled_issues(project, labels)

In [ ]:
if issues:
    print(f"Found {len(issues)} issues with label '<your label>'")
    for issue in issues:
        print(f"Key: {issue['key']}")
        print(f"Summary: {issue['fields']['summary']}")
        print(f"Status: {issue['fields']['status']['name']}")
        print("---")

In [ ]:
issues[1]["key"]

### Getting details from an issue

In [115]:
def format_content_to_markdown(content):
    """Convert Jira's JSON content format to markdown"""
    if not content or not isinstance(content, dict):
        return ""
    
    markdown_text = ""
    
    for item in content.get("content", []):
        # Handle headings
        if item["type"] == "heading":
            level = item["attrs"]["level"]
            heading_text = "".join(
                content.get("text", "") 
                for content in item.get("content", [])
            )
            markdown_text += f"{'#' * level} {heading_text}\n\n"
        
        # Handle paragraphs
        elif item["type"] == "paragraph":
            paragraph_text = ""
            for content in item.get("content", []):
                if content["type"] == "text":
                    # Check for marks (bold, italic, etc)
                    text = content.get("text", "")
                    if "marks" in content:
                        for mark in content["marks"]:
                            if mark["type"] == "strong":
                                text = f"**{text}**"
                    paragraph_text += text
                elif content["type"] == "mention":
                    paragraph_text += f"@{content['attrs'].get('text', '')}"
                elif content["type"] == "emoji":
                    paragraph_text += f":{content['attrs'].get('shortName', '')[1:-1]}:"
                elif content["type"] == "inlineCard":
                    paragraph_text += f"[{content['attrs'].get('url', '')}]"
            markdown_text += f"{paragraph_text}\n\n"
        
        # Handle bullet lists
        elif item["type"] == "bulletList":
            for list_item in item.get("content", []):
                list_content = list_item.get("content", [{}])[0].get("content", [])
                item_text = ""
                for content in list_content:
                    if content.get("type") == "text":
                        item_text += content.get("text", "")
                    elif content.get("type") == "mention":
                        item_text += f"@{content['attrs'].get('text', '')}"
                markdown_text += f"* {item_text}\n"
            markdown_text += "\n"
    
    return markdown_text.strip()

def _get_linked_issues(links):
    return [{
        "key": link.get("outwardIssue", {}).get("key"),
        "summary": link.get("outwardIssue", {}).get("fields", {}).get("summary"),
        "type": link.get("type", {}).get("name"),
        "status": link.get("outwardIssue", {}).get("fields", {}).get("status", {}).get("name")
    } for link in links if "outwardIssue" in link]

def _get_clean_history(history):
    return [{
        "author": item.get("author", {}).get("displayName"),
        "created": item.get("created"),
        "changes": [{
            "field": change.get("field"),
            "from": change.get("fromString"),
            "to": change.get("toString")
        } for change in item.get("items", [])]
    } for item in history]



def get_child_issues(issue_key):
    """Get child issues using JQL search"""
    url = f"{JIRA_URL}/rest/api/3/search"
    jql = f'parent = {issue_key}'
    
    headers = {
        "Accept": "application/json"
    }
    
    params = {
        "jql": jql,
        "fields": "summary,status,assignee,key"
    }
    
    try:
        response = requests.get(
            url,
            headers=headers,
            params=params,
            auth=HTTPBasicAuth(USERNAME, API_TOKEN)
        )
        response.raise_for_status()
        data = response.json()
        
        return [{
            "key": issue.get("key"),
            "summary": issue.get("fields", {}).get("summary"),
            "status": issue.get("fields", {}).get("status", {}).get("name"),
            "assignee": issue.get("fields", {}).get("assignee", {}).get("displayName") if issue.get("fields", {}).get("assignee") else None
        } for issue in data.get("issues", [])]
    except requests.exceptions.RequestException:
        return []

def get_insights(issue_key):
    """Get insights for a specific issue"""
    INSIGHTS_ENDPOINT = f"/rest/api/3/issue/{issue_key}/properties/insight"
    
    try:
        response = requests.get(
            f"{JIRA_URL}{INSIGHTS_ENDPOINT}",
            headers=headers,
            auth=HTTPBasicAuth(USERNAME, API_TOKEN)
        )
        response.raise_for_status()
        
        insights_data = response.json()
        return [{
            "author": insight.get("author", {}).get("displayName"),
            "created": insight.get("created"),
            "content": format_content_to_markdown(insight.get("content", {})),
            "impact": insight.get("impact"),
            "labels": insight.get("labels", [])
        } for insight in insights_data.get("insights", [])]
    except requests.exceptions.RequestException:
        return []


def get_clean_issue_info(issue_key):
    API_ISSUE_ENDPOINT = f"/rest/api/3/issue/{issue_key}"

    headers = {
        "Accept": "application/json"
    }

    params = {
        "fields": "*all",
        "expand": "changelog,renderedFields,insight"
    }

    try:
        response = requests.get(
            f"{JIRA_URL}{API_ISSUE_ENDPOINT}",
            headers=headers,
            params=params,
            auth=HTTPBasicAuth(USERNAME, API_TOKEN)
        )
        response.raise_for_status()
        data = response.json()
        fields = data.get("fields", {})

        # Get linked and child issues
        linked_issues = _get_linked_issues(fields.get("issuelinks", []))
        child_issues = get_child_issues(issue_key)
        labels = fields.get("labels", [])

        # Safely get custom field values
        launch_phase = fields.get("customfield_25484")
        launch_phase_value = launch_phase.get("value") if launch_phase else None

        product = fields.get("customfield_20650", [{}])
        product_value = product[0].get("value") if product and product[0] else None

        business_area = fields.get("customfield_18711")
        business_area_value = business_area.get("value") if business_area else None

        # Safely get teams with default empty list
        teams = fields.get("customfield_18717")
        team_values = [team.get("value") for team in teams if team and team.get("value")] if teams else []

        # Safely get points of contact with default empty list
        contacts = fields.get("customfield_21943")
        contact_list = contacts if isinstance(contacts, list) else []

        clean_data = {
            "basic_info": {
                "key": issue_key,
                "summary": fields.get("summary"),
                "status": fields.get("status", {}).get("name"),
                "assignee": fields.get("assignee", {}).get("displayName") if fields.get("assignee") else None,
                "description": format_content_to_markdown(fields.get("description", {}))
            },
            "project_details": {
                "project_target": fields.get("customfield_18723"),
                "project_start": fields.get("customfield_18722"),
                "launch_phase": launch_phase_value,
                "product": product_value,
                "public_description": fields.get("customfield_18718")
            },
            "team_info": {
                "teams": team_values,
                "data_business_area": business_area_value,
                "points_of_contact": [
                    contact.get("displayName") 
                    for contact in contact_list
                    if contact and contact.get("displayName")
                ]
            },
            "labels": labels,
            "insights": get_insights(issue_key),
            "comments": [{
                "author": comment.get("author", {}).get("displayName"),
                "created": comment.get("created"),
                "body": format_content_to_markdown(comment.get("body", {}))
            } for comment in fields.get("comment", {}).get("comments", [])]
        }

        # Apply the rule for roadmap issues
        if "<yourlabel>" in labels and linked_issues and not child_issues:
            clean_data["child_issues"] = linked_issues
        else:
            clean_data["linked_issues"] = linked_issues
            if child_issues:
                clean_data["child_issues"] = child_issues
        
        return json.dumps(clean_data, indent=2)

    except requests.exceptions.RequestException as e:
        print(f"Error fetching issue details: {e}")
        return None

In [ ]:
issue_info = get_clean_issue_info("DBPD-635")
if issue_info:
    print(issue_info)

### Testing retrieval and weekly saves

In [75]:
import requests
from requests.auth import HTTPBasicAuth
import json
from datetime import datetime
import pandas as pd
import os

In [133]:
def get_linked_issues(issue_key):
    """Get all linked issues that are children of the given issue"""
    API_ISSUE_ENDPOINT = f"/rest/api/3/issue/{issue_key}"
    
    try:
        response = requests.get(
            f"{JIRA_URL}{API_ISSUE_ENDPOINT}",
            headers=headers,
            params={"fields": "issuelinks"},
            auth=HTTPBasicAuth(USERNAME, API_TOKEN)
        )
        response.raise_for_status()
        data = response.json()
        
        links = data.get("fields", {}).get("issuelinks", [])
        return [{
            "key": link.get("outwardIssue", {}).get("key"),
            "type": link.get("type", {}).get("name")
        } for link in links if "outwardIssue" in link]
    except requests.exceptions.RequestException:
        return []

def get_all_related_issues(issue_key, parent_key=None, processed_keys=None):
    """Recursively get all related issues (children and linked)"""
    if processed_keys is None:
        processed_keys = set()
    
    if issue_key in processed_keys:
        return []
    
    processed_keys.add(issue_key)
    all_issues = []
    
    # Get issue details
    issue_info = json.loads(get_clean_issue_info(issue_key))
    issue_info["parent_issue"] = parent_key
    issue_info["extraction_date"] = datetime.now().isoformat()
    all_issues.append(issue_info)
    
    # Get child issues
    children = get_child_issues(issue_key)
    for child in children:
        child_issues = get_all_related_issues(child["key"], issue_key, processed_keys)
        all_issues.extend(child_issues)
    
    # Get linked issues for roadmap items
    if "roadmap-mr-program-2025" in issue_info.get("labels", []):
        linked = get_linked_issues(issue_key)
        for link in linked:
            linked_issues = get_all_related_issues(link["key"], issue_key, processed_keys)
            all_issues.extend(linked_issues)
    
    return all_issues

def get_weekly_jira_data(project_id, labels):
    """Get all issues and their data recursively"""
    all_issues = []
    processed_keys = set()
    
    # Get all parent issues
    parent_issues = get_all_labeled_issues(project_id, labels)
    
    for parent in parent_issues:
        parent_issues = get_all_related_issues(parent["key"], None, processed_keys)
        all_issues.extend(parent_issues)
    
    return all_issues


#============================

def save_weekly_data(issues_data):
    """Save the weekly data with specific columns"""
    week_number = datetime.now().strftime("%Y-W%W")
    folder_path = f"jira_weekly_data/{week_number}"
    os.makedirs(folder_path, exist_ok=True)
    
    # Save raw JSON data
    with open(f"{folder_path}/raw_data.json", "w") as f:
        json.dump(issues_data, f, indent=2)
    
    # Extract and format the data for DataFrame
    formatted_data = []
    for issue in issues_data:
        basic_info = issue.get("basic_info", {})
        project_details = issue.get("project_details", {})
        team_info = issue.get("team_info", {})
        
        formatted_issue = {
            "key": basic_info.get("key"),
            "summary": basic_info.get("summary"),
            "status": basic_info.get("status"),
            "assignee": basic_info.get("assignee"),
            "points_of_contact": ", ".join(team_info.get("points_of_contact", [])),
            "project_target": project_details.get("project_target"),
            "project_start": project_details.get("project_start"),
            "launch_phase": project_details.get("launch_phase"),
            "product": project_details.get("product"),
            "description": basic_info.get("description"),
            "public_description": project_details.get("public_description"),
            "teams": ", ".join(team_info.get("teams", [])),
            "data_business_area": team_info.get("data_business_area"),
            "insights": issue.get("insights"),
            "comments": issue.get("comments"),
            "linked_issues": issue.get("linked_issues"),
            "parent_issue": issue.get("parent_issue"),
            "extraction_date": issue.get("extraction_date"),
            "child_issues": issue.get("child_issues")
        }
        formatted_data.append(formatted_issue)
    
    # Convert to DataFrame
    df = pd.DataFrame(formatted_data)
    
    # Save CSV
    df.to_csv(f"{folder_path}/issues_data.csv", index=False)
    
    return week_number


def compare_weekly_data(current_week, previous_week):
    """Compare data between two weeks"""
    
    try:
        # Load current and previous week data
        current_df = pd.read_csv(f"jira_weekly_data/{current_week}/issues_data.csv")
        previous_df = pd.read_csv(f"jira_weekly_data/{previous_week}/issues_data.csv")
        
        changes = {
            "new_issues": [],
            "closed_issues": [],
            "status_changes": [],
            "description_changes": [],
            "comment_changes": []
        }
        
        # Find new and closed issues
        current_keys = set(current_df["basic_info.key"])
        previous_keys = set(previous_df["basic_info.key"])
        
        changes["new_issues"] = list(current_keys - previous_keys)
        changes["closed_issues"] = list(previous_keys - current_keys)
        
        # Compare status and other changes for existing issues
        for key in current_keys & previous_keys:
            current_issue = current_df[current_df["basic_info.key"] == key].iloc[0]
            previous_issue = previous_df[previous_df["basic_info.key"] == key].iloc[0]
            
            if current_issue["basic_info.status"] != previous_issue["basic_info.status"]:
                changes["status_changes"].append({
                    "key": key,
                    "from": previous_issue["basic_info.status"],
                    "to": current_issue["basic_info.status"]
                })
        
        return changes
        
    except FileNotFoundError:
        return {"error": "Previous week data not found"}
